# GPS Localization for Autonomous Vehicles

## Using Linear Gaussian Systems to Infer Position from Noisy Measurements

This notebook implements the concepts from **Chapter 3.3.4** (Inferring an Unknown Vector) of *Probabilistic Machine Learning* by Kevin Murphy.

### Real-World Scenario

An autonomous vehicle needs to know its precise position to navigate safely. However, GPS measurements are inherently noisy due to:
- Atmospheric interference
- Signal multipath (reflections off buildings)
- Satellite geometry
- Receiver noise

We'll use **Bayesian inference with Linear Gaussian Systems** to:
1. Start with an uncertain prior belief about the vehicle's position
2. Incorporate multiple noisy GPS readings
3. Compute the posterior distribution over the true position
4. Watch uncertainty shrink as more measurements arrive

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

np.random.seed(42)

# Plotting style
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

## 1. The Mathematical Framework

### Prior Distribution

We model the unknown true position $\mathbf{z} \in \mathbb{R}^2$ (latitude, longitude offset in meters) with a Gaussian prior:

$$p(\mathbf{z}) = \mathcal{N}(\mathbf{z} | \boldsymbol{\mu}_z, \boldsymbol{\Sigma}_z)$$

where:
- $\boldsymbol{\mu}_z$ is our initial guess of the position
- $\boldsymbol{\Sigma}_z$ represents our uncertainty (large if we're very unsure)

### Likelihood (Measurement Model)

Each GPS measurement $\mathbf{y}_n$ is a noisy observation of the true position:

$$p(\mathbf{y}_n | \mathbf{z}) = \mathcal{N}(\mathbf{y}_n | \mathbf{z}, \boldsymbol{\Sigma}_y)$$

where $\boldsymbol{\Sigma}_y$ is the measurement noise covariance.

### Key Insight: Combining Multiple Measurements

For $N$ independent measurements, the likelihood of all observations can be represented as a single Gaussian evaluated at their average $\bar{\mathbf{y}}$:

$$p(\mathcal{D}|\mathbf{z}) = \prod_{n=1}^{N} \mathcal{N}(\mathbf{y}_n | \mathbf{z}, \boldsymbol{\Sigma}_y) \propto \mathcal{N}\left(\bar{\mathbf{y}} | \mathbf{z}, \frac{1}{N}\boldsymbol{\Sigma}_y\right)$$

This is powerful: more measurements effectively reduce the noise variance by a factor of $N$!

## 2. Bayes Rule for Gaussians

Using the Linear Gaussian System framework (Equation 3.37 in the book), the posterior is:

$$p(\mathbf{z} | \mathbf{y}_1, \ldots, \mathbf{y}_N) = \mathcal{N}(\mathbf{z} | \hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\Sigma}})$$

where:

$$\hat{\boldsymbol{\Sigma}}^{-1} = \boldsymbol{\Sigma}_z^{-1} + N\boldsymbol{\Sigma}_y^{-1}$$

$$\hat{\boldsymbol{\mu}} = \hat{\boldsymbol{\Sigma}} \left( \boldsymbol{\Sigma}_y^{-1} (N\bar{\mathbf{y}}) + \boldsymbol{\Sigma}_z^{-1} \boldsymbol{\mu}_z \right)$$

### Intuition:
- **Posterior precision** = Prior precision + (Number of measurements × Measurement precision)
- **Posterior mean** = Weighted combination of prior mean and data mean, weighted by their precisions

In [3]:
def bayesian_update_gaussian(mu_prior, Sigma_prior, measurements, Sigma_measurement):
    """
    Perform Bayesian update for a Gaussian prior with Gaussian likelihood.
    
    This implements Bayes rule for Gaussians (Equation 3.37):
    - Posterior precision = Prior precision + N * Measurement precision
    - Posterior mean = Posterior covariance * (Measurement precision * sum(y) + Prior precision * mu_prior)
    
    Parameters:
    -----------
    mu_prior : ndarray (D,)
        Prior mean
    Sigma_prior : ndarray (D, D)
        Prior covariance
    measurements : ndarray (N, D)
        N measurements, each of dimension D
    Sigma_measurement : ndarray (D, D)
        Measurement noise covariance
    
    Returns:
    --------
    mu_posterior : ndarray (D,)
        Posterior mean
    Sigma_posterior : ndarray (D, D)
        Posterior covariance
    """
    N = len(measurements)
    D = len(mu_prior)
    
    # Compute precision matrices (inverse of covariance)
    Lambda_prior = np.linalg.inv(Sigma_prior)
    Lambda_measurement = np.linalg.inv(Sigma_measurement)
    
    # Posterior precision: Λ_post = Λ_prior + N * Λ_measurement
    Lambda_posterior = Lambda_prior + N * Lambda_measurement
    
    # Posterior covariance
    Sigma_posterior = np.linalg.inv(Lambda_posterior)
    
    # Sum of measurements
    y_sum = np.sum(measurements, axis=0)
    
    # Posterior mean: μ_post = Σ_post * (Λ_measurement * Σy + Λ_prior * μ_prior)
    mu_posterior = Sigma_posterior @ (Lambda_measurement @ y_sum + Lambda_prior @ mu_prior)
    
    return mu_posterior, Sigma_posterior

## 3. Visualization Utilities

We'll visualize uncertainty using **confidence ellipses**. For a 2D Gaussian, the contours of constant probability form ellipses. The shape is determined by the eigendecomposition of the covariance matrix (see Section 3.2.2 on Mahalanobis distance).

In [ ]:
def plot_confidence_ellipse(mu, Sigma, ax, n_std=2.0, **kwargs):
    """
    Plot a confidence ellipse for a 2D Gaussian.
    
    The ellipse represents the region where the Mahalanobis distance
    from the mean is less than n_std standard deviations.
    
    For n_std=2, this contains approximately 86.5% of the probability mass.
    For n_std=1, this contains approximately 39.3% of the probability mass.
    """
    # Eigendecomposition of covariance matrix
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    
    # Sort by eigenvalue (largest first)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Compute angle of rotation (angle of first eigenvector)
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    
    # Width and height are 2 * n_std * sqrt(eigenvalue)
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    
    ellipse = Ellipse(xy=mu, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)
    
    return ellipse


def plot_gaussian_2d(mu, Sigma, ax, n_std=2.0, label=None, color='blue', alpha=0.3):
    """
    Plot a 2D Gaussian as a confidence ellipse with its mean marked.
    """
    # Plot mean
    ax.scatter(mu[0], mu[1], c=color, s=100, marker='x', linewidths=2, zorder=5)
    
    # Plot confidence ellipse
    plot_confidence_ellipse(mu, Sigma, ax, n_std=n_std, 
                           facecolor=color, alpha=alpha, 
                           edgecolor=color, linewidth=2, label=label)

## 4. Simulation Setup: Autonomous Vehicle GPS

Let's set up our scenario:
- The vehicle is at a **true position** (which we pretend not to know)
- We have a **prior belief** about where it might be (from the last known position)
- The GPS provides **noisy measurements** with known noise characteristics

In [ ]:
# =============================================================================
# SCENARIO SETUP
# =============================================================================

# True vehicle position (unknown to the algorithm)
# Units: meters from some reference point
z_true = np.array([50.0, 30.0])

# Prior belief: We think the vehicle is near the origin
# with significant uncertainty (standard deviation of 40m in each direction)
mu_prior = np.array([0.0, 0.0])
Sigma_prior = np.array([[1600.0, 0.0],    # 40^2 = 1600 variance
                        [0.0, 1600.0]])

# GPS measurement noise
# Typical consumer GPS has ~5m accuracy (1-sigma)
# We model slightly anisotropic noise: more uncertainty in x (East-West)
# due to satellite geometry (HDOP effects)
sigma_x = 8.0   # meters (East-West uncertainty)
sigma_y = 5.0   # meters (North-South uncertainty)
Sigma_gps = np.array([[sigma_x**2, 0.0],
                      [0.0, sigma_y**2]])

print("=" * 60)
print("GPS LOCALIZATION SCENARIO")
print("=" * 60)
print(f"\nTrue vehicle position: ({z_true[0]:.1f}m, {z_true[1]:.1f}m)")
print(f"\nPrior belief:")
print(f"  Mean: ({mu_prior[0]:.1f}m, {mu_prior[1]:.1f}m)")
print(f"  Std dev: ({np.sqrt(Sigma_prior[0,0]):.1f}m, {np.sqrt(Sigma_prior[1,1]):.1f}m)")
print(f"\nGPS measurement noise:")
print(f"  Std dev: ({sigma_x:.1f}m East-West, {sigma_y:.1f}m North-South)")

## 5. Generate Simulated GPS Measurements

In [ ]:
# Generate N noisy GPS measurements
N_measurements = 20

# Each measurement: y_n = z_true + noise
# noise ~ N(0, Sigma_gps)
noise = np.random.multivariate_normal(np.zeros(2), Sigma_gps, size=N_measurements)
measurements = z_true + noise

print(f"Generated {N_measurements} GPS measurements:")
print(f"\nFirst 5 measurements (x, y in meters):")
for i in range(min(5, N_measurements)):
    print(f"  Measurement {i+1}: ({measurements[i, 0]:6.2f}, {measurements[i, 1]:6.2f})")

print(f"\nSample mean of all measurements: ({measurements.mean(axis=0)[0]:.2f}, {measurements.mean(axis=0)[1]:.2f})")
print(f"True position:                   ({z_true[0]:.2f}, {z_true[1]:.2f})")

## 6. Sequential Bayesian Updates

Let's watch how our belief evolves as we incorporate measurements one by one. This demonstrates the key insight: **uncertainty decreases with more data**.

In [ ]:
def run_sequential_updates(mu_prior, Sigma_prior, measurements, Sigma_measurement):
    """
    Perform sequential Bayesian updates and store history.
    """
    history = {
        'mu': [mu_prior.copy()],
        'Sigma': [Sigma_prior.copy()],
        'n_measurements': [0]
    }
    
    for n in range(1, len(measurements) + 1):
        mu_post, Sigma_post = bayesian_update_gaussian(
            mu_prior, Sigma_prior, 
            measurements[:n], 
            Sigma_measurement
        )
        history['mu'].append(mu_post.copy())
        history['Sigma'].append(Sigma_post.copy())
        history['n_measurements'].append(n)
    
    return history

# Run sequential updates
history = run_sequential_updates(mu_prior, Sigma_prior, measurements, Sigma_gps)

In [ ]:
# Visualize the evolution of belief
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Select specific time points to visualize
time_points = [0, 1, 3, 5, 10, 20]

for idx, n in enumerate(time_points):
    ax = axes[idx]
    
    # Plot true position
    ax.scatter(z_true[0], z_true[1], c='green', s=200, marker='*', 
               label='True Position', zorder=10, edgecolors='black', linewidths=1)
    
    # Plot measurements seen so far
    if n > 0:
        ax.scatter(measurements[:n, 0], measurements[:n, 1], 
                   c='red', alpha=0.5, s=50, label=f'GPS Measurements (n={n})')
    
    # Plot current belief (posterior)
    mu_current = history['mu'][n]
    Sigma_current = history['Sigma'][n]
    
    plot_gaussian_2d(mu_current, Sigma_current, ax, n_std=2.0,
                    label='95% Confidence Region', color='blue', alpha=0.3)
    
    # Plot posterior mean
    ax.scatter(mu_current[0], mu_current[1], c='blue', s=100, marker='x',
               linewidths=3, label='Estimated Position')
    
    # Formatting
    ax.set_xlim(-80, 100)
    ax.set_ylim(-60, 80)
    ax.set_xlabel('East-West (m)')
    ax.set_ylabel('North-South (m)')
    ax.set_title(f'After {n} measurements', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Bayesian GPS Localization: Evolution of Position Estimate', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Analyzing Uncertainty Reduction

Let's quantify how uncertainty decreases with more measurements. According to theory:

$$\hat{\boldsymbol{\Sigma}}^{-1} = \boldsymbol{\Sigma}_z^{-1} + N\boldsymbol{\Sigma}_y^{-1}$$

For a weak prior ($\boldsymbol{\Sigma}_z^{-1} \approx 0$), we get:

$$\hat{\boldsymbol{\Sigma}} \approx \frac{1}{N}\boldsymbol{\Sigma}_y$$

The posterior variance decreases as $1/N$ - the **law of large numbers** in action!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

n_values = history['n_measurements']

# Extract variances over time
var_x = [Sigma[0, 0] for Sigma in history['Sigma']]
var_y = [Sigma[1, 1] for Sigma in history['Sigma']]

# Standard deviations
std_x = np.sqrt(var_x)
std_y = np.sqrt(var_y)

# Theoretical prediction (for large N, ignoring prior)
n_theory = np.arange(1, N_measurements + 1)
std_x_theory = sigma_x / np.sqrt(n_theory)
std_y_theory = sigma_y / np.sqrt(n_theory)

# Plot 1: Standard deviation over time
ax1 = axes[0]
ax1.plot(n_values, std_x, 'b-o', label='σ_x (East-West)', markersize=4)
ax1.plot(n_values, std_y, 'r-s', label='σ_y (North-South)', markersize=4)
ax1.plot(n_theory, std_x_theory, 'b--', alpha=0.5, label='Theory: σ/√N (x)')
ax1.plot(n_theory, std_y_theory, 'r--', alpha=0.5, label='Theory: σ/√N (y)')
ax1.set_xlabel('Number of Measurements')
ax1.set_ylabel('Standard Deviation (m)')
ax1.set_title('Uncertainty Reduction', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, N_measurements)

# Plot 2: Estimation error over time
ax2 = axes[1]
errors = [np.linalg.norm(mu - z_true) for mu in history['mu']]
ax2.plot(n_values, errors, 'g-o', markersize=4)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('Number of Measurements')
ax2.set_ylabel('Distance from True Position (m)')
ax2.set_title('Estimation Error', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, N_measurements)

# Plot 3: Posterior mean trajectory
ax3 = axes[2]
mu_trajectory = np.array(history['mu'])
ax3.plot(mu_trajectory[:, 0], mu_trajectory[:, 1], 'b-', alpha=0.5, linewidth=1)
ax3.scatter(mu_trajectory[:, 0], mu_trajectory[:, 1], 
            c=n_values, cmap='Blues', s=50, edgecolors='black', linewidths=0.5)
ax3.scatter(z_true[0], z_true[1], c='green', s=200, marker='*', 
            label='True Position', zorder=10, edgecolors='black')
ax3.scatter(mu_prior[0], mu_prior[1], c='red', s=100, marker='o', 
            label='Prior Mean', zorder=10)
ax3.set_xlabel('East-West (m)')
ax3.set_ylabel('North-South (m)')
ax3.set_title('Posterior Mean Trajectory', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()

# Print final results
print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"\nTrue position:      ({z_true[0]:.2f}, {z_true[1]:.2f}) m")
print(f"Prior mean:         ({mu_prior[0]:.2f}, {mu_prior[1]:.2f}) m")
print(f"Posterior mean:     ({history['mu'][-1][0]:.2f}, {history['mu'][-1][1]:.2f}) m")
print(f"\nPrior std dev:      ({np.sqrt(Sigma_prior[0,0]):.2f}, {np.sqrt(Sigma_prior[1,1]):.2f}) m")
print(f"Posterior std dev:  ({std_x[-1]:.2f}, {std_y[-1]:.2f}) m")
print(f"\nUncertainty reduction factor: {np.sqrt(Sigma_prior[0,0])/std_x[-1]:.1f}x")
print(f"Final estimation error: {errors[-1]:.2f} m")

## 8. The Role of the Prior

Let's explore how different priors affect the posterior. This demonstrates the Bayesian tradeoff between:
- **Strong prior**: More weight on prior belief, slower adaptation to data
- **Weak prior**: More weight on data, faster convergence to MLE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Different prior strengths
prior_configs = [
    ('Strong Prior\n(σ=10m)', 100.0),    # Strong: small variance
    ('Medium Prior\n(σ=40m)', 1600.0),   # Medium: our default
    ('Weak Prior\n(σ=200m)', 40000.0),   # Weak: large variance
]

# Use only first 5 measurements for clearer visualization
n_show = 5
measurements_subset = measurements[:n_show]

for idx, (title, prior_var) in enumerate(prior_configs):
    ax = axes[idx]
    
    # Set up prior
    Sigma_prior_test = np.array([[prior_var, 0.0], [0.0, prior_var]])
    
    # Compute posterior
    mu_post, Sigma_post = bayesian_update_gaussian(
        mu_prior, Sigma_prior_test, measurements_subset, Sigma_gps
    )
    
    # Plot true position
    ax.scatter(z_true[0], z_true[1], c='green', s=200, marker='*', 
               label='True Position', zorder=10, edgecolors='black')
    
    # Plot measurements
    ax.scatter(measurements_subset[:, 0], measurements_subset[:, 1], 
               c='red', alpha=0.6, s=50, label='Measurements')
    
    # Plot MLE (sample mean)
    mle = measurements_subset.mean(axis=0)
    ax.scatter(mle[0], mle[1], c='orange', s=150, marker='d', 
               label='MLE (Sample Mean)', zorder=8, edgecolors='black')
    
    # Plot prior
    plot_gaussian_2d(mu_prior, Sigma_prior_test, ax, n_std=2.0,
                    color='gray', alpha=0.2)
    ax.scatter(mu_prior[0], mu_prior[1], c='gray', s=100, marker='o', 
               label='Prior Mean')
    
    # Plot posterior
    plot_gaussian_2d(mu_post, Sigma_post, ax, n_std=2.0,
                    color='blue', alpha=0.4)
    ax.scatter(mu_post[0], mu_post[1], c='blue', s=100, marker='x',
               linewidths=3, label='Posterior Mean')
    
    ax.set_xlim(-80, 100)
    ax.set_ylim(-80, 80)
    ax.set_xlabel('East-West (m)')
    ax.set_ylabel('North-South (m)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle(f'Effect of Prior Strength (using {n_show} measurements)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Anisotropic Measurement Noise

In real GPS systems, measurement uncertainty is often **anisotropic** (different in different directions) due to:
- Satellite geometry (HDOP, VDOP)
- Urban canyons blocking certain satellites
- Atmospheric conditions

Let's see how the posterior adapts to anisotropic noise.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Different noise configurations
noise_configs = [
    ('Isotropic Noise\n(σ_x = σ_y = 6m)', np.array([[36, 0], [0, 36]])),
    ('More East-West Noise\n(σ_x=12m, σ_y=4m)', np.array([[144, 0], [0, 16]])),
    ('Correlated Noise\n(ρ = 0.7)', np.array([[64, 0.7*8*6], [0.7*8*6, 36]])),
]

# Weak prior for this demonstration
Sigma_prior_weak = np.array([[10000, 0], [0, 10000]])

for idx, (title, Sigma_noise) in enumerate(noise_configs):
    ax = axes[idx]
    
    # Generate measurements with this noise
    np.random.seed(123)  # Same seed for comparison
    noise_samples = np.random.multivariate_normal(np.zeros(2), Sigma_noise, size=15)
    meas = z_true + noise_samples
    
    # Compute posterior
    mu_post, Sigma_post = bayesian_update_gaussian(
        mu_prior, Sigma_prior_weak, meas, Sigma_noise
    )
    
    # Plot true position
    ax.scatter(z_true[0], z_true[1], c='green', s=200, marker='*', 
               label='True Position', zorder=10, edgecolors='black')
    
    # Plot measurements
    ax.scatter(meas[:, 0], meas[:, 1], c='red', alpha=0.5, s=40, label='Measurements')
    
    # Plot posterior
    plot_gaussian_2d(mu_post, Sigma_post, ax, n_std=2.0,
                    color='blue', alpha=0.4)
    ax.scatter(mu_post[0], mu_post[1], c='blue', s=100, marker='x',
               linewidths=3, label='Posterior Mean')
    
    # Show measurement noise ellipse (scaled for visibility)
    plot_confidence_ellipse(z_true, Sigma_noise, ax, n_std=1.0,
                           facecolor='none', edgecolor='red', 
                           linestyle='--', linewidth=2)
    
    ax.set_xlim(20, 80)
    ax.set_ylim(0, 60)
    ax.set_xlabel('East-West (m)')
    ax.set_ylabel('North-South (m)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Effect of Measurement Noise Structure', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Key Takeaways

### From the Mathematics:

1. **Bayes Rule for Gaussians** (Eq. 3.37) gives us closed-form posterior updates:
   - Posterior precision = Prior precision + N × Measurement precision
   - Posterior mean = Precision-weighted average of prior mean and data

2. **Uncertainty decreases as 1/N**: With N measurements, posterior variance ≈ measurement variance / N

3. **The posterior adapts to noise structure**: Anisotropic measurement noise leads to anisotropic posteriors

### Practical Implications for GPS Localization:

1. **More measurements = better accuracy**: Averaging GPS readings improves position estimates

2. **Prior knowledge helps**: Starting with a reasonable prior (e.g., last known position) speeds up convergence

3. **Know your sensors**: Understanding noise characteristics allows optimal sensor fusion

4. **Real systems extend this**: Kalman filters (covered in later chapters) add motion models for tracking moving vehicles

## 11. Extension: Comparison with Maximum Likelihood

Let's compare Bayesian inference with simple Maximum Likelihood Estimation (MLE), which just takes the sample mean.

In [ ]:
# Compare Bayesian estimate vs MLE for different amounts of data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reset random seed and generate new measurements
np.random.seed(42)
N_total = 50
all_measurements = z_true + np.random.multivariate_normal(np.zeros(2), Sigma_gps, size=N_total)

# Track errors
n_range = range(1, N_total + 1)
bayes_errors = []
mle_errors = []

for n in n_range:
    # Bayesian estimate
    mu_bayes, _ = bayesian_update_gaussian(mu_prior, Sigma_prior, all_measurements[:n], Sigma_gps)
    bayes_errors.append(np.linalg.norm(mu_bayes - z_true))
    
    # MLE (sample mean)
    mu_mle = all_measurements[:n].mean(axis=0)
    mle_errors.append(np.linalg.norm(mu_mle - z_true))

# Plot errors
ax1 = axes[0]
ax1.plot(n_range, bayes_errors, 'b-', label='Bayesian Estimate', linewidth=2)
ax1.plot(n_range, mle_errors, 'r--', label='MLE (Sample Mean)', linewidth=2)
ax1.set_xlabel('Number of Measurements')
ax1.set_ylabel('Distance from True Position (m)')
ax1.set_title('Bayesian vs MLE: Estimation Error', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Show small-sample behavior
ax2 = axes[1]
ax2.plot(n_range[:15], bayes_errors[:15], 'b-o', label='Bayesian Estimate', linewidth=2, markersize=6)
ax2.plot(n_range[:15], mle_errors[:15], 'r--s', label='MLE (Sample Mean)', linewidth=2, markersize=6)
ax2.set_xlabel('Number of Measurements')
ax2.set_ylabel('Distance from True Position (m)')
ax2.set_title('Small Sample Behavior (n ≤ 15)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observation:")
print("- With few measurements, Bayesian and MLE can differ significantly")
print("- With many measurements, both converge to similar estimates")
print("- Bayesian approach provides uncertainty quantification (the posterior covariance)")

## Summary

This notebook demonstrated **Linear Gaussian Systems** (Chapter 3.3.4) through GPS localization:

| Concept | Application |
|---------|-------------|
| Unknown vector **z** | True vehicle position |
| Prior p(**z**) | Initial position belief |
| Measurements **y**_n | GPS readings |
| Likelihood p(**y**\|**z**) | GPS noise model |
| Posterior p(**z**\|**y**) | Updated position estimate |

The key equations (Bayes Rule for Gaussians, Eq. 3.37):

$$\hat{\boldsymbol{\Sigma}}^{-1} = \boldsymbol{\Sigma}_z^{-1} + N\boldsymbol{\Sigma}_y^{-1}$$

$$\hat{\boldsymbol{\mu}} = \hat{\boldsymbol{\Sigma}} \left( \boldsymbol{\Sigma}_y^{-1} (N\bar{\mathbf{y}}) + \boldsymbol{\Sigma}_z^{-1} \boldsymbol{\mu}_z \right)$$

This forms the foundation for more advanced techniques like **Kalman filtering** for tracking moving targets.